# SpamGuard DW — Exploratory Data Analysis

Dokuz Eylul University · Graduation project · Emre Akkaya (2026)

This notebook explores the star schema, inspects label distributions, and walks through model diagnostics.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB = Path('../db/SpamGuard_DW.db')
conn = sqlite3.connect(DB)

overview = pd.read_sql('SELECT * FROM v_spam_overview', conn)
overview

## 1 · Row counts across layers

In [ ]:
tables = ['stg_email_raw','stg_spam_labels','DimSender','DimDate','DimSubject','FactEmail']
counts = {t: conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in tables}
pd.Series(counts, name='rows').sort_values(ascending=False)

## 2 · Monthly time-series (volume + spam rate)

In [ ]:
df = pd.read_sql('''
    SELECT d.year, d.month, COUNT(*) AS total,
           SUM(CASE WHEN f.is_spam=1 THEN 1 ELSE 0 END) AS spam
    FROM FactEmail f JOIN DimDate d ON d.date_key=f.date_key
    WHERE d.year BETWEEN 2000 AND 2002
    GROUP BY d.year, d.month ORDER BY d.year, d.month
''', conn)
df['ym'] = df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2)
df['rate'] = 100 * df['spam'] / df['total']

fig, ax1 = plt.subplots(figsize=(12,4))
ax2 = ax1.twinx()
ax1.bar(df['ym'], df['total'], alpha=.3, color='#8b5cf6', label='volume')
ax2.plot(df['ym'], df['rate'], color='#ef4444', marker='o', label='spam rate %')
ax1.set_xticklabels(df['ym'], rotation=60, fontsize=8)
ax1.set_ylabel('emails'); ax2.set_ylabel('spam %')
plt.title('Monthly volume & spam rate — Enron corpus')
plt.tight_layout(); plt.show()

## 3 · Internal vs external spam rates

In [ ]:
df2 = pd.read_sql('''
    SELECT s.is_internal, COUNT(*) total,
           SUM(CASE WHEN f.is_spam=1 THEN 1 ELSE 0 END) spam
    FROM FactEmail f JOIN DimSender s ON s.sender_key=f.sender_key
    GROUP BY s.is_internal
''', conn)
df2['rate'] = 100 * df2['spam'] / df2['total']
df2['label'] = df2['is_internal'].map({1:'internal @enron.com', 0:'external'})
df2[['label','total','spam','rate']]

## 4 · Top 10 spam-heavy domains

In [ ]:
pd.read_sql('''
    SELECT domain, total_emails, spam_count, spam_rate_pct
    FROM v_spam_by_domain
    WHERE total_emails >= 50 AND spam_rate_pct >= 50
    ORDER BY total_emails DESC LIMIT 10
''', conn)

## 5 · Feature signal: uppercase ratio vs spam

In [ ]:
df3 = pd.read_sql('''
    SELECT CASE WHEN sj.uppercase_ratio >= 0.5 THEN 'HIGH'
                WHEN sj.uppercase_ratio >= 0.2 THEN 'MED'
                ELSE 'low' END AS upper_bucket,
           COUNT(*) total,
           SUM(CASE WHEN f.is_spam=1 THEN 1 ELSE 0 END) spam
    FROM FactEmail f JOIN DimSubject sj ON sj.subject_key=f.subject_key
    GROUP BY upper_bucket
''', conn)
df3['rate'] = 100 * df3['spam'] / df3['total']
df3

## 6 · Model diagnostics

In [ ]:
import joblib, numpy as np
m = joblib.load('../models/spam_model.pkl')
vec = m.named_steps['tfidf']; clf = m.named_steps['clf']
diff = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
feats = vec.get_feature_names_out()

print('Top 15 spam indicators:')
for i in diff.argsort()[::-1][:15]:
    print(f'  {feats[i]:30s}  +{diff[i]:.2f}')
print('\nTop 15 ham indicators:')
for i in diff.argsort()[:15]:
    print(f'  {feats[i]:30s}  {diff[i]:.2f}')